In [1]:
import os
import pandas as pd
from elasticsearch import Elasticsearch, helpers
import re
import json
import requests
from snowflake.snowpark.session import Session
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# VERSION = 28_11_24
VERSION = "27_01_25"

In [3]:
env_data = {"dev":{
  "sbertURL": "https://gst-sbert-model-dev.southcentralus.inference.ml.azure.com/score",
  "sbertAPIKey": "<SBERT_API_KEY_DEV>",
  "elasticSearchURL": "https://elast-gst-nprd-ussc-01.es.privatelink.southcentralus.azure.elastic-cloud.com:9243",
  "elasticSearchPassword": "<ELASTICSEARCH_PASSWORD_DEV>",
  "nerURL": "https://gst-ner-endpoint-dev.southcentralus.inference.ml.azure.com/score",
  "nerAPIKey": "<AML_ENDPOINT_KEY_DEV>",
  },
"qa":{
  "sbertURL": "https://gst-sbert-model-qa.southcentralus.inference.ml.azure.com/score",
  "sbertAPIKey":"<SBERT_API_KEY_QA>",
  "elasticSearchURL": "https://elast-gst-tst-ussc-01.es.privatelink.southcentralus.azure.elastic-cloud.com:9243",
  "elasticSearchPassword": "<ELASTICSEARCH_PASSWORD_QA>",
  "nerURL": "https://gst-ner-endpoint-qa.southcentralus.inference.ml.azure.com/score",
  "nerAPIKey": "<AML_ENDPOINT_KEY_QA>",
  },
"prod":{
  "sbertURL": "https://gst-sbert-model-prod.southcentralus.inference.ml.azure.com/score",
  "sbertAPIKey":"<SBERT_API_KEY_PROD>",
  "elasticSearchURL": "https://elast-gst-prd-ussc-01.es.privatelink.southcentralus.azure.elastic-cloud.com:9243",
  "elasticSearchPassword": "<ELASTICSEARCH_PASSWORD_PROD>",
  "nerURL": "https://gst-ner-endpoint-prod.southcentralus.inference.ml.azure.com/score",
  "nerAPIKey": "<AML_ENDPOINT_KEY_PROD>",
  }}
env="dev"
def get_es():
    ELASTIC_PASSWORD = env_data[env]['elasticSearchPassword']
    es = Elasticsearch(
        env_data[env]['elasticSearchURL'],
        basic_auth=("elastic", ELASTIC_PASSWORD))
    print(es.info())
    return es

ES = get_es()

 
connection_params = {
    # "user": "rahul.s_contractor@celanese.com",
    "user": "ajay.kulkarni_contractor@celanese.com",
    "authenticator": "externalbrowser",
    "account": "celanese-celanytics.privatelink",
    "warehouse": "reporting_wh",
    "database": "analytics_dev",
    "schema": "snowpark",
    "role": "data_analyst_gst"  }
snowpark_session = Session.builder.configs(connection_params).create()

{'name': 'instance-0000000010', 'cluster_name': 'eecb414cd4044754b13fc3adefd89c27', 'cluster_uuid': '3yRqD83mT_2OIPt6w_pxsA', 'version': {'number': '8.16.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'fc1fb13693e87881046baa93e2cf1f4caf2fd58b', 'build_date': '2024-12-12T10:08:52.873963388Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}
Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/7a3c88ff-a5f6-449d-ac6d-e8e3aa508e37/saml2?SAMLRequest=pZPNctowFEZfxaOubcnmNxpMxgmTlhYIE6CdZKfK16AiS64kY3j7CBNm0kWy6coa6Vzp6H7y6PZYyuAAxgqtUhRHBAWguM6F2qZos34IhyiwjqmcSa0gRSew6HY8sqyUFc1qt1NP8LcG6wK%2FkbK0XUhRbRTVzApL

In [4]:
connection_string = eval(os.getenv('connection_string'))
snowflake_connection_string = connection_string['ml-gst-dev-usscc-01']

parts = snowflake_connection_string.split("//")[1].split("/")  
account = ".".join(parts[0].split('.')[:2]) 
user = parts[1].split('user=')[1].split('&')[0] 
password = parts[1].split('password=')[1].split('&')[0] 
database = parts[1].split('db=')[1].split('&')[0]
warehouse = parts[1].split('warehouse=')[1].split('&')[0]  
role = parts[1].split('role=')[1]

engine = create_engine(URL(
    account = account,
    user = user,
    password = password,
    # database = 'ANALYTICS_QA', #database,
    database = 'ANALYTICS_DEV', #database,
    schema = 'gst_curated',
    warehouse = warehouse,
    role = role
))
cur = engine.connect()



def read_data_from_snowflake_table(cur,query):
    df = pd.read_sql(query, cur)
    return df

### Get out of scope data

In [5]:
out_of_scope_brands = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_BRANDS').to_pandas()["BRAND"].unique().tolist()
out_of_scope_polymers = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_POLYMERS').to_pandas()["POLYMER"].unique().tolist()
out_of_scope_grades = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_GRADES').to_pandas()["GRADE"].unique().tolist()
out_of_scope_fillers = snowpark_session.table('GST_CURATED.OUT_OF_SCOPE_FILLERS').to_pandas()["FILLER"].unique().tolist()

## Brand

In [ ]:
out_of_scope_brands

In [ ]:
out_of_scope_brands_cleaned = []
for brand in out_of_scope_brands:
    out_of_scope_brands_cleaned.append(re.sub("(\s+)", " ", re.sub(r'\W+', '', brand).strip().lower()))
        
print(out_of_scope_brands_cleaned)

## Polymer

In [8]:
out_of_scope_polymers

['(EVAC)',
 'SMAH',
 'SBS',
 'Copolyester',
 'PA1010',
 'PTT',
 'SEBS',
 'PS',
 'PVDF',
 'TPO',
 'PAMXD6',
 'PE-LD',
 '(AEM)',
 'PA6T/6I',
 'PEEK',
 'PA12',
 'Paste',
 'ASA']

In [9]:
out_of_scope_polymers_cleaned = []
for polymer in out_of_scope_polymers:
    polymer = re.sub(r'^\((.*)\)$', r'\1', polymer)
    out_of_scope_polymers_cleaned.append(re.sub("(\s+)", " ", polymer.strip().lower()))

out_of_scope_polymers_cleaned

['evac',
 'smah',
 'sbs',
 'copolyester',
 'pa1010',
 'ptt',
 'sebs',
 'ps',
 'pvdf',
 'tpo',
 'pamxd6',
 'pe-ld',
 'aem',
 'pa6t/6i',
 'peek',
 'pa12',
 'paste',
 'asa']

## Filler

In [10]:
out_of_scope_fillers

['Natural organic fiber', 'Glass filler', 'Whisker']

In [11]:
out_of_scope_fillers_cleaned = []
for filler in out_of_scope_fillers:    
    out_of_scope_fillers_cleaned.append(re.sub("(\s+)", " ", filler.strip().lower()))
    
out_of_scope_fillers_cleaned= [i for i in out_of_scope_fillers_cleaned if i!='glass filler']
out_of_scope_fillers_cleaned

['natural organic fiber', 'whisker']

## Grades

In [ ]:
out_of_scope_grades

In [ ]:
out_of_scope_grades_cleaned = []
remove_suffixes = ['(ok1)', '(extra)', '(f1 apply gy/bk only)', '(r&d sample)', '(short version)', '(complete data)', '(before new yc)', '(extrusion)', '(condensed data)', '(r&d trial sample)', '(spcl)', '(eu)', '(us)', '(inte)', '(inte', '(condensed)', '(simplified)', '(dev)', '(old version)', '(old)', '(developmental)', ' - asia', ' - europe', ' - americas']
remove_suffixes += [
    '(f1 apply GY/BK only)',
    '(Zytel HTNFE150099 NC010)',
    '(Zytel HTNFE150099 GY220)',
    '(Zytel HTNFE150099 BK241)',
    '(Zytel FE160000 NC010)',
    '(0.3mm)',
    '(340)',
    '(831)',
    '(2)',
    '(htnfxj8562)',
]
# remove_suffixes = []

for grade in out_of_scope_grades:
    grade = re.sub(r'^\((.*)\)$', r'\1', grade)
    for suffix in remove_suffixes:
        grade = re.sub(r'{}$'.format(re.escape(suffix.lower())), '', grade.lower()) 
    out_of_scope_grades_cleaned.append(re.sub("(\s+)", " ", grade.replace('®', '').replace('™', '').strip()))

out_of_scope_grades_cleaned

In [ ]:
# validating
for i in ['ptfe', 'grade', 'product', 'pellet']:
    for g in out_of_scope_grades_cleaned:
        if i in g:
            print(g)

# ignoring for model training
ignore_grades = [
    'vamac ultra dx carbon black compound',
    'vamac ultra ls for faster cure',
    'mcm low loss greentape 943c2',
    'vamac vmx5020 70-80 shore a',
    'vamac ultra dx for injection molding',
    'sofprene 183bsu865 bianco micro anti uv',
    'sofprene 180nl0190 neutro micro extra',
    'vamac ultra ls for heat resistance',
    'sofprene 384as2470 argento metal.',
    'vamac ultra dx for compression set',
    'vamac ultra dx for wire & cables',
    'sofprene 183bsl255 bianco micro s light',
    'sofprene 183bsu265 bianco micro anti uv',
    'vamac ultra dx for pressureless cure',
    'vamac ultra ls for long term sealing',
    'vamac vmx5015 70-80 shore a',
    'crastin fr2000tc wt001 - orig recipe',
    'vamac vmx5015 65-70 shore a',
    'vamac ultra dx with peroxide',
    'sofprene base 38y/60 ambra rotor gum',
    'vamac ultra ls / ultra ip for transmission',
    'vamac vmx3123 accelerated compound',
    'sofprene 280nu0243 neutro anti uv',
    'vamac ultra ht-or carbon black compound',
    'sofprene base 21153/62 panna butter an',
    'vamac vmx5020 65-70 shore a',
    'sofprene base 21406/62 pure white',
    'vamac ultra dx for fluid resistance',
    'vamac vmx3123 for injection molding',
    'sofprene 183bsl460 bianco micro s light',
    'fortron test only',
    "celanyl test only",
    "cnl test only",
    'sofprene 189nsl060 nero micro s light',   
]
for i in ["china", "ford", "impact", "industrial", "bmw", "product", 'pellet']:
    for g in out_of_scope_grades_cleaned:
        if i in g:
            print(g)
            ignore_grades.append(g)

out_of_scope_grades_cleaned = [x.rstrip("-./") for x in out_of_scope_grades_cleaned if x not in ignore_grades]

pp copo ca/20 nero industrial grade
product x lgc90-qx
product x mt12r01
product x uv90z
product x c 9021 gv1/30 gt
product x c 13031 xf
product x s 27063
product x wr90z
product x c 9021 gv1/10
product x m270
product x s 27072 ws 10/1570
product x uv270z
product x mc90-hm
product x c 27021 ast
product x mt24u01
product x mt12u03
product x c 27021
product x uv25z
product x s 9244
product x m15hp
product x cp15x
product x gb10
product x s 27064
product x uv140lg
product x c 2521
product x lx90gc15
product x c 13031
product x lx90z
product x c 13031 k
product x c 9021 gv1/20 xgm
product x lw90bsx
product x c 9021 gv1/30
product x lw90-s2
product x tx90
product x c 9021 tf5
product x m90
product x s 9363
product x c 9021 gv3/30
product x ec140cf10
product x c 9021 sw
product x gb25
product x c 9021 gv3/10
product x s 9362
product x ec140xf
product x m25
product x c 13021
product x cf802
product x fk 1:25 & m90-07 color masterbatches
product x m25ae
product x gc20
product x tx90plus
produc

In [15]:
# terms = [' only', 'colored', 'pure', 'white', 'recipe', 'metal', ' micro', 'extra', 'super', 'economico', 'arr', 'light', 'gray', ' for', 'long', 'term', 'sealing', 'preliminary', 'snow', 'white', 'med', 'with', 'farbig', 'super', 'econ', 'shore', 'copy', 'backup', 'low', 'loss', 'greentape', 'low', 'loss', 'low', 'compound']
# for g in out_of_scope_grades_cleaned:
#     if any(i in g for i in terms):
#         print(g)

In [16]:
brackets_words = []
bracket_grades = []
for i in out_of_scope_grades_cleaned:
    if len(i.split("(")) > 1:
        if 'inte' in i.split("(")[-1]:
            print(i)
        brackets_words.append('(' + i.split("(")[-1])
        bracket_grades.append(i)

brackets_words = list(set(brackets_words))
brackets_words

gur sl 180 (internal)


['(rif. 389t330)',
 '(specific)',
 '(internal)',
 '(rif. c/495e) talco',
 '(f25-ow)',
 '(sil)',
 '(fu2020) lof2',
 '(sovrastampaggio)',
 '(fu2015)',
 '(rif.369b077)',
 '(nx)',
 '(talco)',
 '(fe270086) bk416',
 '(pant. 18-1755)',
 '(218v100)',
 '(fu2050)',
 '(s)',
 '(n010)',
 '(rif.381aw10)',
 '(7c45) x3',
 '(9t310)',
 '(rif. 3b123)',
 '(fu2020)',
 '(fu2025)',
 '(experimental)',
 '(9t380)',
 '(fu2020) lof',
 '(van)']

In [17]:
# brackets_data = {"LETTERS/NUMBERS": sorted(bracket_grades)}

In [18]:
# with open("OOS_Grades_with_brackets.json", "w") as fp:
#     json.dump(brackets_data, fp, indent=3)

In [19]:
# region_suffix = [' - asia', ' - europe', ' - americas']
# region_suffix_grades = []
# for i in out_of_scope_grades_cleaned:
#     for r in region_suffix:
#         if r in i.lower():
#             region_suffix_grades.append(i)

# region_suffix_grades

In [20]:
for i in out_of_scope_grades_cleaned:
    if 'Zytel HTNFE150099 GY220'.lower() in i:
        print(i)
        break

In [21]:
for i in out_of_scope_grades_cleaned:
    if 'htnfxj8562' in i:
        print(i)
        break

In [22]:
for i in out_of_scope_grades_cleaned:
    if 'product' in i:
        print(i)
        break

In [23]:
placeholder_grades_cleaned = []
for i in out_of_scope_grades_cleaned:
    if '..' in i:
        placeholder_grades_cleaned.append(re.sub("(\s+)", " ", i.replace('...', ' ').replace('..', ' ')))

placeholder_grades_cleaned

['laprene 8m0 d50',
 'laprene 8e0 d50',
 'laprene 8mfg3 a40',
 'laprene 8md a65',
 'laprene 8mg a30',
 'laprene 8ed a50',
 'laprene 8mgd a60',
 'laprene 8m0 a40',
 'laprene 8m0 d40',
 'laprene 8mpe a75',
 'laprene 8mc a70',
 'laprene 8mg b50',
 'forflex 7mr a75',
 'laprene 8mf a75',
 'laprene 8e5 a73',
 'laprene 8mcx35 a75',
 'laprene 8ec a65',
 'laprene 8mg a40',
 'laprene 8mg a55',
 'laprene 8mg a90',
 'laprene 8ec a65b',
 'laprene 8mc5u a70',
 'laprene 8mf a70',
 'laprene 8m3 a70',
 'laprene 8mcf3d a75',
 'laprene 8m0 a50',
 'laprene 8m3sr a67',
 'laprene 8m3 a45',
 'laprene 8e3dx a42',
 'laprene 8efdx a39',
 'laprene 8ecz5 a75',
 'laprene 8e0 a66',
 'laprene 830 540',
 'laprene 8e0 a65',
 'laprene 8mf3 a70',
 'laprene 8m35 d40',
 'laprene 8ef a45',
 'laprene 8m0 d60',
 'laprene 8ec a90',
 'laprene 8e0 a40',
 'laprene 830 561',
 'laprene 8m5 a65',
 'laprene 8mc3 a55',
 'laprene 8e0 d40',
 'laprene 8m0 a80',
 'laprene 8mf a90',
 'laprene 8m3sr a70',
 'laprene 8ecd5 a65',
 'laprene 8m

In [24]:
print(len(out_of_scope_grades_cleaned))
out_of_scope_grades_cleaned = [x for x in out_of_scope_grades_cleaned if '..' not in x]
out_of_scope_grades_cleaned += placeholder_grades_cleaned
print(len(out_of_scope_grades_cleaned))

6678
6678


In [25]:
for i in out_of_scope_grades_cleaned:
    if '  ' in i:
        print(i)

In [26]:
col = 'brand'
ignore_syn=[]
ignore_key=[]
positives = []
# SYNONYM data from dev database i.e. "analytics_dev". Refer to "DEFINED_NAME" and "SYNONYMS".
synonym_df = read_data_from_snowflake_table(cur,"""select * from SYNONYM""")
synonym_df.columns = [x.upper() if x.islower() else x for x in synonym_df.columns]
synonym_df = synonym_df.apply(lambda x: x.str.lower())

synonym_df = synonym_df[synonym_df['TYPE'] == col]

synonym_df2 = synonym_df[["DEFINED_NAME", "SYNONYMS"]].copy()
synonym_df2 = synonym_df2.dropna().reset_index(drop=True)
# synonym_df2 = synonym_df2[synonym_df2['SYNONYMS'].apply(lambda x : False if ";" in x else True)].reset_index(drop=True)
synonym_df2 = synonym_df2[synonym_df2['SYNONYMS'].apply(lambda x : True if ";" in x else False)].reset_index(drop=True)
brand_synonyms = synonym_df2.to_dict(orient='records')
brands_with_synonym = [item['DEFINED_NAME'] for item in brand_synonyms]
brand_synonyms  = {item['DEFINED_NAME']: item['SYNONYMS'] for item in brand_synonyms}

for k in brand_synonyms:
    if ";" in brand_synonyms[k]:
        brand_synonyms[k] = [x.strip() for x in  brand_synonyms[k].split(';') if x!=k]

# grade with brand synonyms
grade_names_with_brand_synonym = []
for i in brands_with_synonym:
   brand_pattern = fr'^{re.escape(i)}\b'
   for grade in out_of_scope_grades_cleaned:
      if re.match(brand_pattern, grade):
         for s in brand_synonyms[i]:
            grade_names_with_brand_synonym.append(re.sub(brand_pattern, s, grade))

out_of_scope_grades_cleaned.extend(grade_names_with_brand_synonym)
out_of_scope_grades_cleaned = list(set(out_of_scope_grades_cleaned))
print(len(out_of_scope_grades_cleaned))

13980


In [27]:
outOfScopeData = {
    "grades":list(set(out_of_scope_grades_cleaned)),
    "brands":list(set(out_of_scope_brands_cleaned)),
    "polymers":list(set(out_of_scope_polymers_cleaned)),
    "fillers":list(set(out_of_scope_fillers_cleaned)),
}

In [28]:
with open(f"./outOfScopeData_{VERSION}.json", "w") as fp:
    json.dump(outOfScopeData , fp, indent=3)